# Recruitment Analytics & Hiring Intelligence
## Initial data exploration and timestamp integrity audit

**Purpose:** Examine six related synthetic recruitment tables, verify their keys and event chronology, and document timestamp exceptions **without overwriting or deleting original records**.

**Source:** [Mindweave — Job Postings & Applications](https://huggingface.co/datasets/mindweave/job-postings-applications) (synthetic data).  
**Approach:** Observe → Validate → Investigate → Document → Decide. This notebook does **not** assume that unusual interview outcomes are data-entry errors.

**Important:** The six original CSV files should be placed alongside this notebook. Run cells in order. The findings below were observed in the previous working session and should be reconfirmed when the full notebook runs.

### 1. Load the source tables

Each DataFrame represents a separate business entity or transaction. The total number of records across tables is **not** the number of unique candidates. The original CSVs are read but never written to.

In [1]:
# Import pandas for tabular data analysis.
import pandas as pd

# Import NumPy for readable conditional classification.
import numpy as np

# Import display for notebook tables.
from IPython.display import display

# Load job application records from the original CSV.
applications = pd.read_csv("applications.csv")
# Load candidate profiles from the original CSV.
candidates = pd.read_csv("candidates.csv")
# Load company profiles from the original CSV.
companies = pd.read_csv("companies.csv")
# Load interview events from the original CSV.
interviews = pd.read_csv("interviews.csv")
# Load advertised jobs from the original CSV.
jobs = pd.read_csv("jobs.csv")
# Load offer decisions and acceptance dates from the original CSV.
offers = pd.read_csv("offers.csv")

# Group the six source tables for repeated audits.
datasets = {"Applications": applications, "Candidates": candidates,
            "Companies": companies, "Interviews": interviews,
            "Jobs": jobs, "Offers": offers}

# Display row and column counts for each table.
display(pd.DataFrame({name: {"rows": len(df), "columns": len(df.columns)}
                      for name, df in datasets.items()}).T)

,rows,columns
Applications,15000,7
Candidates,9000,6
Companies,320,5
Interviews,7196,5
Jobs,2000,11
Offers,938,6


### 2. Inspect schema and database integrity

Expected relationships: `companies → jobs → applications ← candidates`; `applications → interviews` and `applications → offers`. `interview_id` is unique; `application_id` is **expected to repeat** in `interviews` because one application may have multiple interviews.

We verify primary-key uniqueness and existence, foreign-key references, and missing values. These checks do not confirm that business-process sequencing is valid.

In [2]:
# Define the expected unique identifier in every table.
primary_keys = {"Applications": "application_id", "Candidates": "candidate_id",
                "Companies": "company_id", "Interviews": "interview_id",
                "Jobs": "job_id", "Offers": "offer_id"}

# Measure key uniqueness and completeness without changing any records.
key_audit = pd.DataFrame([
    {"table": name, "duplicate_primary_keys": int(df[key].duplicated().sum()),
     "missing_primary_keys": int(df[key].isna().sum())}
    for name, key in primary_keys.items()
    for df in [datasets[name]]
])

# Display primary-key audit results.
display(key_audit)

# Define child and parent key combinations for referential integrity checks.
foreign_keys = [
    ("Applications", "job_id", "Jobs", "job_id"),
    ("Applications", "candidate_id", "Candidates", "candidate_id"),
    ("Jobs", "company_id", "Companies", "company_id"),
    ("Interviews", "application_id", "Applications", "application_id"),
    ("Offers", "application_id", "Applications", "application_id")
]

# Count missing or unmatched foreign keys in each relationship.
fk_audit = pd.DataFrame([
    {"relationship": f"{child}.{child_key} → {parent}.{parent_key}",
     "invalid_references": int((~datasets[child][child_key].isin(datasets[parent][parent_key])).sum())}
    for child, child_key, parent, parent_key in foreign_keys
])

# Display referential-integrity results.
display(fk_audit)

# Summarize missing values per field across the six tables.
missing_audit = pd.concat({name: df.isna().sum() for name, df in datasets.items()},
                          names=["table", "column"]).rename("missing_count")

# Display fields with missing values, or an empty table if none exist.
display(missing_audit[missing_audit > 0].to_frame())

,table,duplicate_primary_keys,missing_primary_keys
0,Applications,0,0
1,Candidates,0,0
2,Companies,0,0
3,Interviews,0,0
4,Jobs,0,0
5,Offers,0,0


,relationship,invalid_references
0,Applications.job_id → Jobs.job_id,0
1,Applications.candidate_id → Candidates.candida...,0
2,Jobs.company_id → Companies.company_id,0
3,Interviews.application_id → Applications.appli...,0
4,Offers.application_id → Applications.applicati...,0


,,missing_count
table,column,
Offers,accepted_at,239


**Previously observed:** All primary and foreign keys passed. Only `offers.accepted_at` had missing values (239), associated with offers marked `accepted=False`. Missing values are not automatically data-quality faults; their business meaning must be checked.

### 3. Convert date types and validate acceptance-status consistency

Timestamp fields initially load as strings. `errors="coerce"` marks unparseable values as `NaT` so they remain visible to the audit. We count any unexpected parsing failures rather than silently treating them as valid dates.

In [3]:
# Define source date fields in their corresponding tables.
date_fields = [(applications, "applied_at"), (jobs, "posted_at"),
               (interviews, "scheduled_at"), (offers, "offered_at"),
               (offers, "accepted_at")]

# Convert each date field to datetime, retaining missing dates as NaT.
for df, column in date_fields:
    # Record how many nonmissing strings existed before conversion.
    original_nonmissing = int(df[column].notna().sum())
    # Parse date strings without inventing a substitute date.
    df[column] = pd.to_datetime(df[column], errors="coerce")
    # Report values lost specifically because date parsing failed.
    print(f"{column}: parse failures = {original_nonmissing - int(df[column].notna().sum())}")

# Cross-tabulate accepted status against missing acceptance date.
acceptance_audit = pd.crosstab(offers["accepted"], offers["accepted_at"].isna(),
                              rownames=["accepted"], colnames=["accepted_at missing"])

# Display whether missing acceptance dates are consistent with status.
display(acceptance_audit)

applied_at: parse failures = 0
posted_at: parse failures = 0
scheduled_at: parse failures = 0
offered_at: parse failures = 0
accepted_at: parse failures = 0


accepted_at missing,False,True
accepted,,
False,0,239
True,699,0


### 4. Offer chronology and transparent exception flag

A negative duration is a verifiable **chronology exception**: the recorded acceptance timestamp precedes the recorded offer timestamp. It does **not** tell us which event timestamp is wrong or whether earlier/revised offers existed. In an earlier exploratory version, we briefly created `Updated_accept` and replaced erroneous dates with the offer date. That adjustment was **rejected and removed** because it assumed same-day acceptance without evidence.

Here, `accepted_at` and `offered_at` remain unchanged. The derived `validation` label identifies the issue without implying that a date was corrected.

In [4]:
# Calculate elapsed days using the unchanged source offer and acceptance timestamps.
offers["acceptance_days"] = (offers["accepted_at"] - offers["offered_at"]).dt.total_seconds() / 86400

# Flag accepted offers whose recorded acceptance occurred before the offer.
chronology_exception = offers["accepted"] & (offers["accepted_at"] < offers["offered_at"])

# Flag expected missing acceptance timestamps on unaccepted offers.
not_applicable = (~offers["accepted"]) & offers["accepted_at"].isna()

# Flag chronological consistency among accepted offers with valid date entries.
valid_acceptance = offers["accepted"] & offers["accepted_at"].notna() & ~chronology_exception

# Assign audit labels without modifying any source dates.
offers["validation"] = np.select(
    [chronology_exception, not_applicable, valid_acceptance],
    ["chronology_exception", "not_applicable", "valid_date"],
    default="review_required"
)

# Display the number of offers in each audit category.
display(offers["validation"].value_counts(dropna=False).to_frame("records"))

# Summarize the original acceptance durations for accepted offers only.
display(offers.loc[offers["accepted"], "acceptance_days"].describe().to_frame("days"))

# Confirm that the provisional adjusted column is not part of this analysis.
print("Updated_accept exists:", "Updated_accept" in offers.columns)

,records
validation,
valid_date,482
not_applicable,239
chronology_exception,217


,days
count,699.000000
mean,6.244635
std,13.961530
min,-27.000000
25%,-4.000000
50%,7.000000
75%,16.000000
max,38.000000


Updated_accept exists: False


**Previously observed:** 938 offers; 699 accepted; 239 not accepted. Among accepted offers, 217 chronology exceptions (31.0%) and 482 dates ordered correctly. The original duration ranged from −27 to 38 days. **Do not use negative response times as observed business KPIs**, and do not assume the remaining positive response times accurately represent real-world behaviour in synthetic data.

### 5. Cross-table chronology checks

We check job posting before application, application before interview, and application before offer. We also identify interviews *scheduled* after an offer. Those are **post-offer interview events**, not proven errors: `scheduled_at` is not evidence of an actual meeting or its cause. A repeated `application_id` in interviews is valid and does not mean duplicate interview records.

In [5]:
# Connect applications with their job posting dates using the job foreign key.
job_app_timeline = applications.merge(jobs[["job_id", "posted_at"]], on="job_id", how="left", validate="many_to_one")

# Connect interviews to their corresponding application dates.
interview_app_timeline = interviews.merge(applications[["application_id", "applied_at"]], on="application_id", how="left", validate="many_to_one")

# Connect offers to their corresponding application dates.
offer_app_timeline = offers.merge(applications[["application_id", "applied_at"]], on="application_id", how="left", validate="many_to_one")

# Associate each interview event with offer dates from the same application.
interview_offer_timeline = interviews.merge(offers[["application_id", "offered_at"]], on="application_id", how="inner")

# Identify scheduled interview events later than an offer date.
post_offer_interviews = interview_offer_timeline.loc[interview_offer_timeline["scheduled_at"] > interview_offer_timeline["offered_at"]].copy()

# Summarize the four chronology checks without changing source records.
chronology_audit = pd.Series({
    "applications_before_job_posting": int((job_app_timeline["applied_at"] < job_app_timeline["posted_at"]).sum()),
    "interviews_before_application": int((interview_app_timeline["scheduled_at"] < interview_app_timeline["applied_at"]).sum()),
    "offers_before_application": int((offer_app_timeline["offered_at"] < offer_app_timeline["applied_at"]).sum()),
    "interviews_scheduled_after_offer": len(post_offer_interviews),
    "applications_with_post_offer_interviews": post_offer_interviews["application_id"].nunique()
}, name="records")

# Display cross-table chronology counts.
display(chronology_audit.to_frame())

# Inspect recorded outcomes associated with post-offer interview events.
display(post_offer_interviews["result"].value_counts().to_frame("interview_events"))

,records
applications_before_job_posting,0
interviews_before_application,0
offers_before_application,0
interviews_scheduled_after_offer,171
applications_with_post_offer_interviews,124


,interview_events
result,
pass,120
fail,31
hold,20


**Previously observed:** 
<li>0 interviews before application; 
<li>0 offers before application; 
<li>**171** post-offer interview events across 
<li>**124** applications. Their results were 120 `pass`, 31 `fail`, and 20 `hold`. 

Job-posting versus application ordering is a **new check** and must be read from the newly executed output. Post-offer interviews are flagged for context, not automatically corrected or discarded.

### 6. Investigate repeated time-of-day components

Our core hypothesis: related recruitment events may have been created from a shared time-of-day with offsets in whole days. We test the *hour:minute:second* component, not whether the full timestamps are identical.

For unaccepted offers, the missing acceptance timestamp is intentionally ignored. For accepted offers, all three time components must match. Interview rows are compared with their corresponding application time.

In [6]:
# Reuse the offer-to-application join with the original timestamp columns.
all_offer_times = offer_app_timeline.copy()

# Check matching application and offer time-of-day values.
matching_offer_time = all_offer_times["applied_at"].dt.time == all_offer_times["offered_at"].dt.time

# Check acceptance time when present, allowing legitimately missing acceptance timestamps.
matching_accept_time = all_offer_times["accepted_at"].isna() | (all_offer_times["offered_at"].dt.time == all_offer_times["accepted_at"].dt.time)

# Require both comparisons to pass for an offer record.
all_offer_times["same_time"] = matching_offer_time & matching_accept_time

# Compare the interview time-of-day with its application time-of-day.
interview_app_timeline["same_time"] = interview_app_timeline["scheduled_at"].dt.time == interview_app_timeline["applied_at"].dt.time

# Summarize the time-component matches across offer and interview events.
time_audit = pd.DataFrame({
    "event_type": ["offers", "interviews"],
    "matching": [int(all_offer_times["same_time"].sum()), int(interview_app_timeline["same_time"].sum())],
    "total": [len(all_offer_times), len(interview_app_timeline)]
})

# Calculate the matching fraction for each event type.
time_audit["matching_pct"] = (100 * time_audit["matching"] / time_audit["total"]).round(2)

# Display the complete time-component audit.
display(time_audit)

# Summarize the same time-component pattern specifically for chronology exceptions.
exception_time_audit = all_offer_times.loc[all_offer_times["validation"] == "chronology_exception", "same_time"]

# Display matching and total exception counts.
print("Chronology exceptions with matching times:", int(exception_time_audit.sum()), "/", len(exception_time_audit))

,event_type,matching,total,matching_pct
0,offers,938,938,100.0
1,interviews,7196,7196,100.0


Chronology exceptions with matching times: 217 / 217


**Previously observed:** matching time components for **938/938 offers**, **7,196/7,196 interviews**, and **217/217 acceptance chronology exceptions**. This is powerful evidence of **systematic timestamp construction** in synthetic data, but it does *not* establish the precise generator algorithm or prove which individual dates are incorrect.

### 7. Two small additional checks

These add value without increasing project scope:

1. Are the event offsets from `applied_at` always **whole days**? This tests the proposed shared-clock/whole-day-offset mechanism directly.
2. Is an application associated with more than one offer? The source allows an `offer_id` primary key, but does not guarantee one offer per application. This helps us understand whether offer revision histories are even represented.

The checks below are diagnostic only; they do not change any records.

In [7]:
# Compute offer time differences relative to application submission in seconds.
offer_offset_seconds = (all_offer_times["offered_at"] - all_offer_times["applied_at"]).dt.total_seconds()

# Compute interview time differences relative to application submission in seconds.
interview_offset_seconds = (interview_app_timeline["scheduled_at"] - interview_app_timeline["applied_at"]).dt.total_seconds()

# Compute acceptance differences only for accepted offers with a recorded date.
accepted_offer_rows = all_offer_times.loc[all_offer_times["accepted"] & all_offer_times["accepted_at"].notna()]

# Compare acceptance date to the original application timestamp.
acceptance_offset_seconds = (accepted_offer_rows["accepted_at"] - accepted_offer_rows["applied_at"]).dt.total_seconds()

# Count any offsets not divisible by exactly one day (86,400 seconds).
whole_day_audit = pd.Series({
    "offers_non_whole_day_offsets": int((offer_offset_seconds.mod(86400) != 0).sum()),
    "interviews_non_whole_day_offsets": int((interview_offset_seconds.mod(86400) != 0).sum()),
    "acceptances_non_whole_day_offsets": int((acceptance_offset_seconds.mod(86400) != 0).sum())
}, name="exceptions")

# Display any exceptions to the common-reference whole-day hypothesis.
display(whole_day_audit.to_frame())

# Count offers for each application to check for represented revisions or repeat offers.
offers_per_application = offers.groupby("application_id")["offer_id"].nunique()

# Count applications linked to two or more offers.
print("Applications with multiple offer records:", int((offers_per_application > 1).sum()))

# Show the maximum number of offer records belonging to one application.
print("Maximum offers for one application:", int(offers_per_application.max()))

,exceptions
offers_non_whole_day_offsets,0
interviews_non_whole_day_offsets,0
acceptances_non_whole_day_offsets,0


Applications with multiple offer records: 0
Maximum offers for one application: 1


### 8. Optional case-study lookup — six tables, without flattening interview history

Change only `app_id` and rerun this cell. We keep interviews separate to avoid duplicating application or offer fields when multiple rounds exist. Useful cases from our investigation: **8, 13, 109, 588, 720, 967**. Individual explanations remain hypotheses, not verified facts.

In [8]:
# Choose the application to investigate without changing the source data.
app_id = 8

# Find the selected application record.
selected_application = applications.loc[applications["application_id"] == app_id]

# Stop with a clear error if the selected application is absent.
if selected_application.empty:
    # Raise an informative error rather than silently returning unrelated records.
    raise ValueError(f"Application ID {app_id} does not exist")

# Retrieve the job ID associated with the application.
selected_job_id = selected_application["job_id"].iloc[0]

# Retrieve the candidate ID associated with the application.
selected_candidate_id = selected_application["candidate_id"].iloc[0]

# Retrieve the selected job to identify the company.
selected_job = jobs.loc[jobs["job_id"] == selected_job_id]

# Retrieve the company ID associated with the job.
selected_company_id = selected_job["company_id"].iloc[0]

# Display company profile records.
print("COMPANY")
display(companies.loc[companies["company_id"] == selected_company_id])

# Display job posting records.
print("JOB")
display(selected_job)

# Display the candidate profile records.
print("CANDIDATE")
display(candidates.loc[candidates["candidate_id"] == selected_candidate_id])

# Display the selected application.
print("APPLICATION")
display(selected_application)

# Display every interview round in timestamp order.
print("INTERVIEWS")
display(interviews.loc[interviews["application_id"] == app_id].sort_values("scheduled_at"))

# Display all offers associated with the application, without replacing dates.
print("OFFERS")
display(offers.loc[offers["application_id"] == app_id].sort_values("offered_at"))

COMPANY


,company_id,company_name,industry,company_size,hq_city
188,189,Garcia LLC,healthcare,growth,Seattle


JOB


,job_id,company_id,posted_at,title,role_family,level,location_type,employment_type,salary_low,salary_high,status
1370,1371,189,2025-04-28 01:49:44,Pricing Analyst,data,mid,remote,full_time,126000,159000,open


CANDIDATE


...

,application_id,job_id,candidate_id,applied_at,source,stage,score
7,8,1371,3874,2025-05-16 01:49:44,linkedin,offer,0.426


INTERVIEWS


,interview_id,application_id,round_name,scheduled_at,result
4,5,8,hiring_manager,2025-05-28 01:49:44,pass
5,6,8,recruiter,2025-06-07 01:49:44,hold
6,7,8,final,2025-06-13 01:49:44,pass


OFFERS


,offer_id,application_id,offered_at,base_salary,accepted,accepted_at,acceptance_days,validation
0,1,8,2025-06-24 01:49:44,130303,True,2025-06-01 01:49:44,-23.0,chronology_exception


## Audit conclusions and project decisions

**Established from the previous investigation (to be verified by rerunning):**

- All six table primary keys are complete and unique; tested foreign-key references are valid.
- The 239 missing `accepted_at` values correspond to unaccepted offers, not unexplained gaps.
- Among 699 accepted offers, **217 (31.0%)** have an acceptance timestamp earlier than the recorded offer timestamp.
- **171** interviews are scheduled after associated offer dates, across **124** applications. These events are **not automatically errors**, and repeated application IDs are expected in the interview table.
- In **all 938 offers and 7,196 interview events**, the time-of-day matches the application's time-of-day. This strongly suggests systematically constructed timestamps in the synthetic source.

**Data-governance decisions:**

1. Retain original source records and timestamps; do not delete observations or fabricate acceptance dates.
2. Use `chronology_exception` as a diagnostic label; the former `offer_date_substituted` label was misleading after the provisional adjustment was removed.
3. Do not calculate offer response-time KPIs using negative durations as if they were real elapsed times. Any analysis using valid-looking durations must acknowledge synthetic timestamp limitations and potential bias.
4. Treat post-offer interviews and `fail`/`hold` interview outcomes as events requiring contextual interpretation, not automatic data-entry errors.
5. Keep the business hypothesis (e.g., revised offers or further interview rounds) separate from supported facts. There are no offer revision logs or system audit trails in these six files to verify those stories.

**New results to inspect after execution:** applications before job posting; whether all event differences are whole-day offsets; whether multiple offers can occur for a single application.

**Next phase (separate notebook):** Define focused recruitment business questions only after these verification results are reviewed. This notebook is an audit, not a predictive model or production ATS correction procedure.